# Metodologia Design Science Research (DSR)

**Etapa de Pesquisa (Peffers et al., 2007):**
### 3. Design e Desenvolvimento (Design and Development)

**Objetivo Acadêmico:** Este notebook foca na ingestão de dados de calendário (feriados, pontes, períodos letivos). No contexto do DSR, isso representa a inclusão de variáveis de contexto externas que explicam a variabilidade estrutural da demanda, essencial para a construção de um artefato resiliente e adaptável ao ambiente institucional.


# 01d - Ingestão de Calendários Multi-Ano (PDF)
Este notebook unifica calendários de diferentes anos e formatos em uma série temporal contínua.

In [1]:
import pandas as pd
pd.set_option('future.no_silent_downcasting', True)
import os
import glob
import re
import unicodedata
from pypdfium2 import PdfDocument

# 1. Configurações
DIRETORIO_PDFS = "../data/calendarios/"
SAIDA_CSV = "../data/calendario_consolidado.csv"

def normalizar_texto(s, manter_espacos=True):
    if not isinstance(s, str): return ""
    # Remove acentos e simplifica caracteres
    n = unicodedata.normalize('NFD', s).encode('ascii', 'ignore').decode('utf-8')
    if manter_espacos:
        n = re.sub(r'[^a-zA-Z0-9\s]', ' ', n)
    else:
        n = re.sub(r'[^a-zA-Z0-9]', '', n)
    return n.upper().strip()

def extrair_texto_pdf(caminho):
    try:
        pdf = PdfDocument(caminho)
        return "\n".join([page.get_textpage().get_text_bounded() for page in pdf])
    except: return ""

def safe_timestamp(y, m, d):
    try:
        return pd.Timestamp(year=y, month=m, day=d)
    except ValueError:
        return None

def parse_eventos_universal(texto, ano_ref):
    meses_map = {'JANEIRO': 1, 'FEVEREIRO': 2, 'MARCO': 3, 'ABRIL': 4, 'MAIO': 5, 'JUNHO': 6, 'JULHO': 7, 'AGOSTO': 8, 'SETEMBRO': 9, 'OUTUBRO': 10, 'NOVEMBRO': 11, 'DEZEMBRO': 12}
    eventos = []
    totais_pdf = []
    
    # Unir linhas quebradas (data numa linha, desc na outra)
    linhas = texto.split('\n')
    texto_unido = ""
    for i in range(len(linhas)):
        l = linhas[i].strip()
        if not l: continue
        if l.endswith('-') and i+1 < len(linhas):
            texto_unido += l + " " + linhas[i+1].strip() + "\n"
            linhas[i+1] = ""
        else:
            texto_unido += l + "\n"

    # EXTRAÇÃO DE TOTAIS (Para Conferência)
    texto_norm = normalizar_texto(texto_unido)
    for mes_nome, mes_num in meses_map.items():
        # Busca o total que aparece logo após o nome do mês
        padrao_total = rf"{mes_nome}.*?DIAS LETIVOS NO MES\s+(\d+)"
        matches = re.finditer(padrao_total, texto_norm, re.DOTALL)
        for m in matches:
            totais_pdf.append({'ano': ano_ref, 'mes': mes_num, 'total_pdf': int(m.group(1))})

    # PADRÃO 1: Blocos Mensais
    blocos = re.split(r'(\w+)\s+ATIVIDADES\s+/\s+EVENTOS', texto_unido)
    if len(blocos) > 1:
        for i in range(1, len(blocos), 2):
            mes_num = meses_map.get(normalizar_texto(blocos[i], manter_espacos=False))
            if not mes_num: continue
            for linha in blocos[i+1].split('\n'):
                match = re.search(r'(\d{1,2}(?:\s*(?:a|e)\s*\d{1,2})?)\s*[-–:]\s*(.*)', linha.strip())
                if match:
                    dias_str, desc = match.group(1), match.group(2).strip()
                    nums = re.findall(r'\d+', dias_str)
                    if not desc: continue
                    if ' a ' in dias_str and len(nums) == 2:
                        for d in range(int(nums[0]), int(nums[1]) + 1):
                            ts = safe_timestamp(ano_ref, mes_num, d)
                            if ts: eventos.append({'data': ts, 'evento': desc})
                    else:
                        for d_str in nums:
                            ts = safe_timestamp(ano_ref, mes_num, int(d_str))
                            if ts: eventos.append({'data': ts, 'evento': desc})

    # PADRÃO 2: Formatos explícitos D/M
    matches_intervalo = re.finditer(r'(?:\s+)?(\d{1,2})/(\d{1,2})\s+A\s+(\d{1,2})/(\d{1,2})\s*[:\-–]\s*(.*)', texto_unido.upper())
    for m in matches_intervalo:
        try:
            d1, m1, d2, m2, desc = m.groups()
            start = safe_timestamp(ano_ref, int(m1), int(d1))
            end = safe_timestamp(ano_ref, int(m2), int(d2))
            if start and end:
                for d in pd.date_range(start, end):
                    eventos.append({'data': d, 'evento': desc.strip()})
        except: continue
        
    matches_simples = re.finditer(r'(?<!\d/)(\d{1,2})/(\d{1,2})\s*[:\-–]\s*([^\n]*)', texto_unido.upper())
    for m in matches_simples:
        try:
            d1, m1, desc = m.groups()
            ts = safe_timestamp(ano_ref, int(m1), int(d1))
            if ts: eventos.append({'data': ts, 'evento': desc.strip()})
        except: continue
        
    return pd.DataFrame(eventos), pd.DataFrame(totais_pdf)

# Execução Multi-Ano
arquivos = glob.glob(os.path.join(DIRETORIO_PDFS, "*.pdf"))
todos, todos_totais = [], []
anos = set()

for f in arquivos:
    txt = extrair_texto_pdf(f)
    ano_match = re.search(r'20\d{2}', os.path.basename(f) + txt)
    ano_ref = int(ano_match.group()) if ano_match else 2025
    print(f"📖 Processando: {os.path.basename(f)} (Ano: {ano_ref})")
    df_pdf, df_totais = parse_eventos_universal(txt, ano_ref)
    if not df_pdf.empty:
        todos.append(df_pdf)
        anos.update(df_pdf['data'].dt.year.unique())
    if not df_totais.empty:
        todos_totais.append(df_totais)

if todos:
    # Consolidação: UNIR descrições na mesma data para não perder marcos importantes
    df_unificado = pd.concat(todos).drop_duplicates(subset=['data', 'evento'])
    df_unificado = df_unificado.groupby('data')['evento'].apply(lambda x: ' / '.join(x.unique())).reset_index()
    
    ano_min, ano_max = min(anos), max(anos)
    datas_gap = pd.date_range(start=f"{ano_min}-01-01", end=f"{ano_max}-12-31")
    df_final = pd.DataFrame({'data': datas_gap})
    df_final['dia_semana_num'] = df_final['data'].dt.dayofweek
    
    df_final['evento_padrao'] = 'DIA LETIVO PADRAO'
    df_final.loc[df_final['dia_semana_num'] == 5, 'evento_padrao'] = 'SABADO'
    df_final.loc[df_final['dia_semana_num'] == 6, 'evento_padrao'] = 'DOMINGO'
    
    df_final = pd.merge(df_final, df_unificado, on='data', how='left')
    df_final['evento'] = df_final['evento'].fillna(df_final['evento_padrao'])
    df_final['norm_evento'] = df_final['evento'].apply(normalizar_texto)
    
    # 1. Feriados e Pontos Facultativos
    df_final['eh_feriado'] = df_final['norm_evento'].str.contains("FERIADO|RECESSO|CARNAVAL|CONFRATERNIZACAO|MUNDIAL DA PAZ|DIA DO TRABALHO|INDEPENDENCIA|FINADOS|PROCLAMACAO|NATAL|PONTO FACULTATIVO")
    df_final['vespera_feriado'] = df_final['eh_feriado'].shift(-1).fillna(False)
    
    # 2. Reuniões de Impacto
    df_final['eh_reuniao_impacto'] = df_final['norm_evento'].str.contains("PAIS|FAMILIA|FAMILIAS|REPLANEJAMENTO|CONSELHO DE CLASSE|CONSELHOS|REAVALIACAO")
    
    # 3. Detectar Início e Fim das Etapas
    df_final['etapa'] = 0
    etapas_list = []
    for i in range(1, 5):
        pat_ini = f"INICIO.*(ETAPA {i}|{i}.? ETAPA)"
        pat_fim = f"(ENCERRAMENTO|TERMINO).*(ETAPA {i}|{i}.? ETAPA)"
        mask_ini = df_final['norm_evento'].str.contains(pat_ini, regex=True)
        mask_fim = df_final['norm_evento'].str.contains(pat_fim, regex=True)
        ini_idx = df_final.loc[mask_ini].index.min() if mask_ini.any() else None
        fim_idx = df_final.loc[mask_fim].index.max() if mask_fim.any() else None
        if ini_idx is not None and fim_idx is not None:
            df_final.loc[ini_idx:fim_idx, 'etapa'] = i
            etapas_list.append({'Etapa': f'Etapa {i}', 'Inicio': df_final.loc[ini_idx, 'data'].strftime('%d/%m/%Y'), 'Fim': df_final.loc[fim_idx, 'data'].strftime('%d/%m/%Y')})
    df_etapas_obs = pd.DataFrame(etapas_list)
    
    # 4. Regra de Negócio: tem_refeicao
    mask_inicio_aulas = df_final['norm_evento'].str.contains("INICIO.*(ETAPA 1|1.? ETAPA)", regex=True)
    data_inicio_aulas = df_final.loc[mask_inicio_aulas, 'data'].min() if mask_inicio_aulas.any() else None
    df_final['tem_refeicao'] = True
    if data_inicio_aulas:
        df_final.loc[df_final['data'] < data_inicio_aulas, 'tem_refeicao'] = False
    df_final.loc[df_final['eh_feriado'] | (df_final['dia_semana_num'] >= 5), 'tem_refeicao'] = False
    
    # 5. Eventos Especiais
    df_final['eh_evento_especial'] = 0
    mask_especial = (df_final['tem_refeicao'] == True) & \
                     (df_final['norm_evento'].str.contains("SEMANA|CAMPUS PORTAS ABERTAS")) & \
                     (~df_final['eh_reuniao_impacto'])
    df_final.loc[mask_especial, 'eh_evento_especial'] = 1
    
    # 6. Preparar Exportação
    df_export = df_final[['data', 'evento', 'eh_feriado', 'vespera_feriado', 'eh_reuniao_impacto', 'tem_refeicao', 'eh_evento_especial', 'etapa']].copy()
    bool_cols = ['eh_feriado', 'vespera_feriado', 'eh_reuniao_impacto', 'tem_refeicao', 'eh_evento_especial']
    df_export[bool_cols] = df_export[bool_cols].astype(int)
    df_export.to_csv(SAIDA_CSV, index=False)
    
    # AUDITORIA AUTOMÁTICA
    if todos_totais:
        df_totais_pdf = pd.concat(todos_totais).drop_duplicates()
        df_calc = df_final.groupby([df_final['data'].dt.year.rename('ano'), df_final['data'].dt.month.rename('mes')])['tem_refeicao'].sum().reset_index()
        df_calc.columns = ['ano', 'mes', 'total_calc']
        df_conf = pd.merge(df_totais_pdf, df_calc, on=['ano', 'mes'], how='inner')
        df_conf['status'] = df_conf.apply(lambda x: '✅ OK' if x['total_pdf'] == x['total_calc'] else f'❌ Erro ({x["total_calc"]-x["total_pdf"]}d)', axis=1)
        df_conf.columns = ['Ano', 'Mês', 'Total PDF', 'Total Calculado', 'Status']
    
    print(f"🚀 SUCESSO! Base multi-ano refinada em: {SAIDA_CSV}")
else:
    print("⚠️ Falha crítica: Nenhum dado extraído.")

📖 Processando: CALENDARIO_2025.pdf (Ano: 2025)
🚀 SUCESSO! Base multi-ano refinada em: ../data/calendario_consolidado.csv


C:\Users\miche\AppData\Local\Temp\ipykernel_11604\3339434954.py:152: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_ini = df_final['norm_evento'].str.contains(pat_ini, regex=True)
C:\Users\miche\AppData\Local\Temp\ipykernel_11604\3339434954.py:153: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_fim = df_final['norm_evento'].str.contains(pat_fim, regex=True)
C:\Users\miche\AppData\Local\Temp\ipykernel_11604\3339434954.py:152: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_ini = df_final['norm_evento'].str.contains(pat_ini, regex=True)
C:\Users\miche\AppData\Local\Temp\ipykernel_11604\3339434954.py:153: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the grou

# 7. Sumário da Ingestão de Calendário
Resumo quantitativo dos dias letivos, feriados e eventos extraídos.

In [2]:
# Sumário de Ingestão e Auditoria de Qualidade
import pandas as pd
df_temp = df_export.copy()
df_temp['data'] = pd.to_datetime(df_temp['data'])
total_dias = len(df_temp)
dias_letivos = df_temp['tem_refeicao'].sum()
total_feriados = df_temp['eh_feriado'].sum()
total_reunioes = df_temp['eh_reuniao_impacto'].sum()
eventos_especiais = df_temp['eh_evento_especial'].sum()
primeira_data = df_temp['data'].min().strftime('%d/%m/%Y')
ultima_data = df_temp['data'].max().strftime('%d/%m/%Y')

print(f"--- RELATÓRIO DE INGESTÃO (CALENDÁRIO REFINADO) ---")
print(f"Total de Dias no Período: {total_dias}")
print(f"Total de Dias Letivos (Aulas Pós-Início): {dias_letivos}")
print(f"Total de Feriados/Pontos Facultativos: {total_feriados}")
print(f"Total de Reuniões de Impacto: {total_reunioes}")
print(f"Total de Eventos Especiais: {eventos_especiais}")
print(f"Período: {primeira_data} até {ultima_data}")

# 1. Cronograma de Etapas Acadêmicas
print("\nCronograma de Etapas Acadêmicas:")
if 'df_etapas_obs' in locals() and not df_etapas_obs.empty:
    print(df_etapas_obs.to_string(index=False))
else:
    print("Nenhuma etapa detectada explicitamente.")

# 2. Auditoria de Integridade (PDF vs CSV)
print("\nAuditoria de Integridade (Dias Letivos por Mês):")
if 'df_conf' in locals() and not df_conf.empty:
    display(df_conf)
else:
    print("Informação de totais não disponível no PDF para comparação automática.")

# 3. Visão Detalhada para Conferência Manual
print("\nDetalhamento para Conferência Manual (Dias com Alterações):")
mask_conferencia = (df_temp['eh_feriado']==1) | \
                    (df_temp['eh_reuniao_impacto']==1) | \
                    (df_temp['eh_evento_especial']==1) | \
                    (df_temp['etapa'] > 0) | \
                    (~df_temp['evento'].isin(['DIA LETIVO PADRAO', 'SABADO', 'DOMINGO']))

detalhe_cal = df_temp[mask_conferencia].copy()
detalhe_cal = detalhe_cal[['data', 'etapa', 'evento', 'tem_refeicao', 'eh_feriado', 'eh_reuniao_impacto', 'eh_evento_especial']]
detalhe_cal.columns = ['Data', 'Etapa', 'Evento/Status', 'Letivo', 'Feriado', 'Reunião', 'Evento Esp.']
detalhe_cal['Data'] = detalhe_cal['Data'].dt.strftime('%d/%m/%Y')

display(detalhe_cal.style.apply(lambda x: ['background-color: #e6fffa' if v == 1 else '' for v in x], subset=['Letivo'])
                  .apply(lambda x: ['background-color: #fff5f5' if v == 1 else '' for v in x], subset=['Feriado'])
                  .apply(lambda x: ['background-color: #fffaf0' if v == 1 else '' for v in x], subset=['Reunião'])
                  .apply(lambda x: ['background-color: #e6f4ff' if v == 1 else '' for v in x], subset=['Evento Esp.'])
                  .apply(lambda x: ['font-weight: bold; color: #2c5282' if v > 0 else '' for v in x], subset=['Etapa']))

--- RELATÓRIO DE INGESTÃO (CALENDÁRIO REFINADO) ---
Total de Dias no Período: 365
Total de Dias Letivos (Aulas Pós-Início): 216
Total de Feriados/Pontos Facultativos: 16
Total de Reuniões de Impacto: 13
Total de Eventos Especiais: 21
Período: 01/01/2025 até 31/12/2025

Cronograma de Etapas Acadêmicas:
  Etapa     Inicio        Fim
Etapa 1 19/02/2025 30/04/2025
Etapa 2 02/05/2025 08/07/2025
Etapa 3 31/07/2025 02/10/2025
Etapa 4 03/10/2025 11/12/2025

Auditoria de Integridade (Dias Letivos por Mês):


,Ano,Mês,Total PDF,Total Calculado,Status
0,2025,1,0,0,✅ OK
1,2025,2,8,8,✅ OK
2,2025,3,20,19,❌ Erro (-1d)
3,2025,4,21,20,❌ Erro (-1d)
4,2025,5,22,21,❌ Erro (-1d)
5,2025,5,0,21,❌ Erro (21d)
6,2025,6,23,20,❌ Erro (-3d)
7,2025,7,6,22,❌ Erro (16d)
8,2025,7,0,22,❌ Erro (22d)
9,2025,8,22,21,❌ Erro (-1d)



Detalhamento para Conferência Manual (Dias com Alterações):


,Data,Etapa,Evento/Status,Letivo,Feriado,Reunião,Evento Esp.
0,01/01/2025,0,"Dia mundial da paz / Confraternização Universal- Lei Federal nº 10.607, de 19/12/2002 (feriado federal)",0,1,0,0
1,02/01/2025,0,Férias (25 dias),0,0,0,0
20,21/01/2025,0,Reunião ordinária do NAPNE,0,0,0,0
22,23/01/2025,0,FÉRIAS (25 DIAS),0,0,0,0
23,24/01/2025,0,FÉRIAS (25 DIAS),0,0,0,0
24,25/01/2025,0,FÉRIAS (25 DIAS),0,0,0,0
25,26/01/2025,0,FÉRIAS (25 DIAS),0,0,0,0
26,27/01/2025,0,FÉRIAS (25 DIAS),0,0,0,0
27,28/01/2025,0,FÉRIAS (25 DIAS),0,0,0,0
28,29/01/2025,0,FÉRIAS (25 DIAS),0,0,0,0
